# Module 02 — Exploring CUDA Threads on an NVIDIA T4

This notebook implements vector addition with Numba CUDA, verifies the thread/block arithmetic, and compares several thread-block sizes. Run it on a Lightning AI Studio with a T4.

## Setup

If needed, install the CUDA-enabled Numba components from a terminal:

```bash
pip install numba numba-cuda numpy
```

In [ ]:
import statistics
import time

import numpy as np
from numba import cuda

assert cuda.is_available(), "CUDA is not available; select a GPU runtime."
device = cuda.get_current_device()
name = device.name.decode() if isinstance(device.name, bytes) else device.name
print("GPU:", name)
print("Compute capability:", device.compute_capability)
print("Multiprocessors:", device.MULTIPROCESSOR_COUNT)
print("Threads per block limit:", device.MAX_THREADS_PER_BLOCK)

## The kernel

Each thread computes one output element. The bounds check protects the final partially filled block.

In [ ]:
@cuda.jit
def vector_add(a, b, c, size):
    index = cuda.grid(1)
    if index < size:
        c[index] = a[index] + b[index]

## Launch 1,024 threads for 1,000 elements

With 256 threads per block, the grid requires four blocks:

$$\left\lceil\frac{1000}{256}\right\rceil = 4,\qquad 4\times256 = 1024\text{ threads}$$

In [ ]:
N = 1000
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

A = np.arange(N, dtype=np.float32)
B = np.arange(N, dtype=np.float32) * 10
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array(N, dtype=np.float32)

vector_add[blocks_per_grid, threads_per_block](d_A, d_B, d_C, N)
cuda.synchronize()
C = d_C.copy_to_host()

print("Blocks:", blocks_per_grid)
print("Threads per block:", threads_per_block)
print("Total threads:", blocks_per_grid * threads_per_block)
print("First 10 outputs:", C[:10])
np.testing.assert_allclose(C, A + B)
print("Correctness: PASS")

## Compare thread-block sizes

The total useful work stays constant. Only the launch configuration changes.

In [ ]:
N = 2_000_000
repeats = 30
A = np.ones(N, dtype=np.float32)
B = np.ones(N, dtype=np.float32)
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(d_A)
results = []

for threads in (32, 64, 128, 256, 512, 1024):
    blocks = (N + threads - 1) // threads
    vector_add[blocks, threads](d_A, d_B, d_C, N)
    cuda.synchronize()

    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        vector_add[blocks, threads](d_A, d_B, d_C, N)
        cuda.synchronize()
        samples.append((time.perf_counter() - start) * 1000)

    median_ms = statistics.median(samples)
    results.append((threads, blocks, median_ms))
    print(f"Threads/block: {threads:4d} | Blocks: {blocks:6d} | Time: {median_ms:.4f} ms")

np.testing.assert_allclose(d_C.copy_to_host(), 2.0)

In [ ]:
try:
    import matplotlib.pyplot as plt

    threads = [row[0] for row in results]
    times = [row[2] for row in results]
    plt.figure(figsize=(8, 4))
    plt.plot(threads, times, marker="o")
    plt.xlabel("Threads per block")
    plt.ylabel("Median launch + execution time (ms)")
    plt.title("CUDA Thread Block Size on NVIDIA T4")
    plt.grid(alpha=0.3)
    plt.show()
except ImportError:
    print("Install matplotlib to plot the benchmark results.")

## Interpretation

Different block sizes alter how blocks are scheduled and how per-block resources are allocated. Do not assume that 1,024 threads per block is always faster than 256, or that 32 is always best. The optimal configuration depends on the kernel and device.